In [2]:
from calendar import month_name
from unicodedata import category

import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
import urllib.parse

In [10]:
load_dotenv()
def get_db_connection():
    
    user = os.getenv('MYSQL_USER')
    password = os.getenv('MYSQL_PASSWORD')
    host = os.getenv('MYSQL_HOST')
    port = os.getenv('MYSQL_PORT')
    database = os.getenv('MYSQL_DB')

    safe_password = urllib.parse.quote_plus(password)

    db_url = f"mysql+pymysql://{user}:{safe_password}@{host}:{port}/{database}"
    engine = create_engine(db_url)
    return engine

def query_data(query):
    try:
        engine = get_db_connection()
        df_result = pd.read_sql(query,con=engine)
        return df_result
    except Exception as e:
        print(f"Error in querying data: {e}")
        return None

In [11]:
household_2024_query = """
select exp.newid,exp.seqno,exp.expname,exp.cost_,exp.ref_mo,exp.ref_yr,exp.gift,exp.ucc,exp.cost,household.fam_size,household.family_income_before_tax,household.calibration_weight,household.number_of_earners,household.popsize,household.interview_month,household.interview_year,household.region,household.sex_ref,household.total_expenditure_prior_quarter,household.total_expenditure_current_quarter,household.state,household.imputed_income_before_tax,household.psu,household.division,household.urban from monthly_expenditure exp left join household_2024_metadata household on exp.newid = household.newid where exp.ucc in ('690114','690111','690120','270102','270106','690113',
     '690114','690310','300311','300312','300321','300322','300331','300332','320522','320232','690117','690119','690116',
     '480100','480213','490501','310316','310140','270310','620930','310231','310232','310400','340610','340902','310314',
     '310350','610130','310243','620917','620918','310333','690320','690330','590230','690118','300311','300312','300321',
     '300322','320331','320332','320522','320232','690111','690117','690119','690120','690115','690116','690210','270106','690310',
     '620930','270310','310140','310231','310232','620917','620918','310243','310400','310316','310314','610130','310333','310350') and exp.ref_yr = 2024;
"""

df_2024_data = query_data(household_2024_query)

In [12]:
df_2024_data.head()

,newid,seqno,expname,cost_,ref_mo,ref_yr,gift,ucc,cost,fam_size,...,interview_year,region,sex_ref,total_expenditure_prior_quarter,total_expenditure_current_quarter,state,imputed_income_before_tax,psu,division,urban
0,5348484,17,QADOTHX,E,1,2024,2,270310,3.0,2,...,2024,2.0,1,10609.3333,6501.1667,17.0,170000.0,S23A,3.0,1
1,5356724,10,QADOTHX,E,2,2024,2,270310,4.0,1,...,2024,4.0,1,4515.7500,8258.5000,15.0,91200.0,S49F,9.0,1
2,5357274,14,QADOTHX,E,2,2024,2,270310,3.0,1,...,2024,4.0,1,3666.5000,9154.0000,2.0,83311.0,S49G,9.0,1
3,5358004,25,QADOTHX,E,2,2024,2,270310,5.0,1,...,2024,3.0,2,2597.6667,12292.3333,24.0,145058.9,S35E,5.0,1
4,5360154,23,QADOTHX,E,2,2024,2,270310,8.0,1,...,2024,3.0,2,4582.0833,10772.1667,12.0,44537.8,None,5.0,1


In [13]:
#df_2024_data.drop_duplicates(inplace=True)

#df_2024_data.dropna(inplace = True)

ucc_2024_map ={
    # Telecommunications
    270102: "Cellular phone service",
    270106: "Residential telephone including VOIP",
    270310: "Cable and satellite television services",
    690114: "Computer information services (internet)",
    690116: "Internet services away from home",
    
    # Computing Hardware & Accessories
    690111: "Computers and computer hardware for nonbusiness use",
    690117: "Portable memory",
    690120: "Computer accessories",
    690115: "Personal digital assistants",
    320232: "Telephones and accessories",
    690210: "Telephone answering devices",
    
    # Software & Digital Services
    690119: "Computer software",
    620930: "Online gaming services",
    310400: "Applications, games, and ringtones for handheld devices",
    
    # Computing Services
    690113: "Repair of computer systems for nonbusiness use",
    690310: "Installation of computers",
    
    # Streaming & Digital Media
    310350: "Streaming and downloading audio",
    310243: "Rental, streaming, and downloading videos",
    620917: "Rental of video hardware/accessories",
    620918: "Rental of video software",
    
    # Gaming
    310231: "Video game software",
    310232: "Video game hardware and accessories",
    
    # Audio/Visual Equipment
    310140: "Televisions",
    310316: "Stereos, radios, speakers, and sound components",
    310314: "Personal digital audio players",
    310333: "Accessories and other sound equipment",
    340610: "Repair of televisions, radio, and sound equipment",
    340902: "Rental of televisions",
    690320: "Installation of televisions",
    690330: "Installation of satellite television equipment",
    
    # Musical Instruments
    610130: "Musical instruments and accessories",
    
    # Digital Reading
    590230: "Books, digital books, or book subscriptions",
    690118: "Digital book readers",
    
    # Appliances (Non-Digital - appear to be duplicates/errors in original list)
    300311: "Cooking stoves and ovens (renter)",
    300312: "Cooking stoves and ovens (owned home)",
    300321: "Microwave ovens (renter)",
    300322: "Microwave ovens (owned home)",
    300331: "Portable dishwashers (renter)",
    300332: "Portable dishwashers (owned home)",
    320522: "Portable heating and cooling equipment",
    
    # Vehicle Accessories (Non-Digital - appear to be errors in original list)
    480100: "Vehicle parts, accessories, fluid excluding tires",
    480213: "Parts, equipment, and accessories",
    490501: "Vehicle accessories including labor",

}

df_2024_data['product_description'] = df_2024_data['ucc'].map(ucc_2024_map).fillna(df_2024_data['ucc'])

#df_2024_data.drop(columns=['ALCNO','PUBFLAG','UCCSEQ'],inplace=True)

df_2024_data['gift'] = df_2024_data['gift'].replace({1: True, 2: False}).astype(bool)

     newid  seqno  expname cost_  ref_mo  ref_yr  gift     ucc  cost  \
0  5348484     17  QADOTHX     E       1    2024     2  270310   3.0   
1  5356724     10  QADOTHX     E       2    2024     2  270310   4.0   
2  5357274     14  QADOTHX     E       2    2024     2  270310   3.0   
3  5358004     25  QADOTHX     E       2    2024     2  270310   5.0   
4  5360154     23  QADOTHX     E       2    2024     2  270310   8.0   

   fam_size  ...  region  sex_ref  total_expenditure_prior_quarter  \
0         2  ...     2.0        1                       10609.3333   
1         1  ...     4.0        1                        4515.7500   
2         1  ...     4.0        1                        3666.5000   
3         1  ...     3.0        2                        2597.6667   
4         1  ...     3.0        2                        4582.0833   

   total_expenditure_current_quarter  state  imputed_income_before_tax   psu  \
0                          6501.1667   17.0                   1700

/tmp/ipykernel_62094/3252315015.py:78: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_2024_data['gift'] = df_2024_data['gift'].replace({1: True, 2: False}).astype(bool)


In [14]:
print(df_2024_data.head())

     newid  seqno  expname cost_  ref_mo  ref_yr   gift     ucc  cost  \
0  5348484     17  QADOTHX     E       1    2024  False  270310   3.0   
1  5356724     10  QADOTHX     E       2    2024  False  270310   4.0   
2  5357274     14  QADOTHX     E       2    2024  False  270310   3.0   
3  5358004     25  QADOTHX     E       2    2024  False  270310   5.0   
4  5360154     23  QADOTHX     E       2    2024  False  270310   8.0   

   fam_size  ...  region  sex_ref  total_expenditure_prior_quarter  \
0         2  ...     2.0        1                       10609.3333   
1         1  ...     4.0        1                        4515.7500   
2         1  ...     4.0        1                        3666.5000   
3         1  ...     3.0        2                        2597.6667   
4         1  ...     3.0        2                        4582.0833   

   total_expenditure_current_quarter  state  imputed_income_before_tax   psu  \
0                          6501.1667   17.0                 

In [15]:
df_2024_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14916 entries, 0 to 14915
Data columns (total 26 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   newid                              14916 non-null  int64  
 1   seqno                              14916 non-null  int64  
 2   expname                            14916 non-null  object 
 3   cost_                              14916 non-null  object 
 4   ref_mo                             14916 non-null  int64  
 5   ref_yr                             14916 non-null  int64  
 6   gift                               14916 non-null  bool   
 7   ucc                                14916 non-null  int64  
 8   cost                               14916 non-null  float64
 9   fam_size                           14916 non-null  int64  
 10  family_income_before_tax           14916 non-null  int64  
 11  calibration_weight                 14916 non-null  flo

In [16]:
df_2024_data.describe()

,newid,seqno,ref_mo,ref_yr,ucc,cost,fam_size,family_income_before_tax,calibration_weight,number_of_earners,...,interview_month,interview_year,region,sex_ref,total_expenditure_prior_quarter,total_expenditure_current_quarter,state,imputed_income_before_tax,division,urban
count,1.491600e+04,14916.000000,14916.000000,14916.0,14916.000000,14916.000000,14916.000000,14916.000000,14916.000000,14916.000000,...,14916.000000,14916.0,14588.000000,14916.000000,14916.000000,14916.000000,13649.000000,14916.000000,13853.000000,14916.000000
mean,5.493857e+06,28.188656,1.322204,2024.0,407490.747721,88.305556,2.414119,107476.207093,29192.922495,1.342987,...,2.640990,2024.0,2.734851,1.503285,10400.194891,11563.333956,27.243534,122277.814173,5.432397,1.171896
std,9.665183e+04,17.432160,0.467336,0.0,183231.160263,170.391364,1.324186,116044.033962,12007.353886,1.021632,...,0.479726,0.0,1.049093,0.500006,11432.998846,10626.875286,16.495773,117610.301431,2.566048,0.377303
min,5.343864e+06,1.000000,1.000000,2024.0,270102.000000,1.000000,1.000000,-6083.000000,1592.845000,0.000000,...,2.000000,2024.0,1.000000,1.000000,146.500000,73.250000,1.000000,-26951.900000,1.000000,1.000000
25%,5.425793e+06,17.000000,1.000000,2024.0,270102.000000,25.000000,1.000000,33258.000000,21876.577000,1.000000,...,2.000000,2024.0,2.000000,1.000000,4285.425050,5204.450000,12.000000,42485.100000,3.000000,1.000000
50%,5.553012e+06,26.000000,1.000000,2024.0,310243.000000,61.000000,2.000000,75000.000000,29410.856000,1.000000,...,3.000000,2024.0,3.000000,2.000000,7168.750000,8599.400100,27.000000,86628.000000,5.000000,1.000000
75%,5.587151e+06,36.000000,2.000000,2024.0,690114.000000,104.000000,3.000000,142784.000000,35692.829000,2.000000,...,3.000000,2024.0,4.000000,2.000000,12259.666700,13985.125000,41.000000,155000.000000,8.000000,1.000000
max,5.608061e+06,718.000000,2.000000,2024.0,690320.000000,9000.000000,10.000000,924823.000000,91983.018000,7.000000,...,3.000000,2024.0,4.000000,2.000000,204944.087500,115150.400000,55.000000,949344.200000,9.000000,2.000000


In [17]:
df_2024_data.isna().sum()

newid                                   0
seqno                                   0
expname                                 0
cost_                                   0
ref_mo                                  0
ref_yr                                  0
gift                                    0
ucc                                     0
cost                                    0
fam_size                                0
family_income_before_tax                0
calibration_weight                      0
number_of_earners                       0
popsize                                 0
interview_month                         0
interview_year                          0
region                                328
sex_ref                                 0
total_expenditure_prior_quarter         0
total_expenditure_current_quarter       0
state                                1267
imputed_income_before_tax               0
psu                                  8096
division                          

In [18]:
df_2024_data.dropna(inplace=True)

In [19]:
df_2024_data.drop_duplicates(inplace=True)

In [20]:
df_2024_data.head()

,newid,seqno,expname,cost_,ref_mo,ref_yr,gift,ucc,cost,fam_size,...,region,sex_ref,total_expenditure_prior_quarter,total_expenditure_current_quarter,state,imputed_income_before_tax,psu,division,urban,product_description
0,5348484,17,QADOTHX,E,1,2024,False,270310,3.0,2,...,2.0,1,10609.3333,6501.1667,17.0,170000.0,S23A,3.0,1,Cable and satellite television services
1,5356724,10,QADOTHX,E,2,2024,False,270310,4.0,1,...,4.0,1,4515.7500,8258.5000,15.0,91200.0,S49F,9.0,1,Cable and satellite television services
2,5357274,14,QADOTHX,E,2,2024,False,270310,3.0,1,...,4.0,1,3666.5000,9154.0000,2.0,83311.0,S49G,9.0,1,Cable and satellite television services
3,5358004,25,QADOTHX,E,2,2024,False,270310,5.0,1,...,3.0,2,2597.6667,12292.3333,24.0,145058.9,S35E,5.0,1,Cable and satellite television services
7,5366304,17,QADOTHX,E,1,2024,False,270310,10.0,3,...,3.0,2,3983.3333,8911.6667,48.0,72000.0,S37B,7.0,1,Cable and satellite television services


In [23]:
region_dict ={
    1: "Northeast",
    2: "Midwest",
    3: "South",
    4: "West"
}
df_2024_data['region_description'] = df_2024_data['region'].map(region_dict).fillna(df_2024_data['region'])
df_2024_data.drop(columns=['region'])
df_2024_data.head()

,newid,seqno,expname,cost_,ref_mo,ref_yr,gift,ucc,cost,fam_size,...,sex_ref,total_expenditure_prior_quarter,total_expenditure_current_quarter,state,imputed_income_before_tax,psu,division,urban,product_description,region_description
0,5348484,17,QADOTHX,E,1,2024,False,270310,3.0,2,...,1,10609.3333,6501.1667,17.0,170000.0,S23A,3.0,1,Cable and satellite television services,Midwest
1,5356724,10,QADOTHX,E,2,2024,False,270310,4.0,1,...,1,4515.7500,8258.5000,15.0,91200.0,S49F,9.0,1,Cable and satellite television services,West
2,5357274,14,QADOTHX,E,2,2024,False,270310,3.0,1,...,1,3666.5000,9154.0000,2.0,83311.0,S49G,9.0,1,Cable and satellite television services,West
3,5358004,25,QADOTHX,E,2,2024,False,270310,5.0,1,...,2,2597.6667,12292.3333,24.0,145058.9,S35E,5.0,1,Cable and satellite television services,South
7,5366304,17,QADOTHX,E,1,2024,False,270310,10.0,3,...,2,3983.3333,8911.6667,48.0,72000.0,S37B,7.0,1,Cable and satellite television services,South


In [24]:
population_dict ={
    1: "5+ Million",
    2: "1-5 Million",
    3: "0.5-1.0 Million",
    4: "100-500 Thousands",
    5: "100- Thousands",
    6: "Suppressed"
}
df_2024_data['Population_bucket'] = pd.Categorical(df_2024_data['popsize'].map(population_dict))

0        1
1        3
2        4
3        2
7        1
        ..
14907    2
14908    2
14909    2
14910    2
14911    2
Name: popsize, Length: 6820, dtype: int64


In [31]:
df_2024_data['state'] = df_2024_data['state'].astype("int64")
state_mapping = {
    1: "Alabama",
    2: "Alaska",
    4: "Arizona",
    5: "Arkansas",
    6: "California",
    8: "Colorado",
    9: "Connecticut",
    10: "Delaware",
    11: "District of Columbia",
    12: "Florida",
    13: "Georgia",
    14: "Massachusetts",
    15: "Hawaii",
    16: "Idaho",
    17: "Illinois",
    18: "Indiana",
    19: "Iowa",
    20: "Kansas",
    21: "Kentucky",
    22: "Louisiana",
    23: "Maine",
    24: "Maryland",
    25: "Massachusetts",
    26: "Michigan",
    27: "Minnesota",
    28: "Mississippi",
    29: "Missouri",
    30: "Montana",
    31: "Nebraska",
    32: "Nevada",
    33: "New Hampshire",
    34: "New Jersey",
    35: "New Mexico",
    36: "New York",
    37: "North Carolina",
    39: "Ohio",
    40: "Oklahoma",
    41: "Oregon",
    42: "Pennsylvania",
    44: "Rhode Island",
    45: "South Carolina",
    46: "South Dakota",
    47: "Tennessee",
    48: "Texas",
    49: "Utah",
    51: "Virginia",
    52: "Maryland",
    53: "Washington",
    54: "West Virginia",
    55: "Wisconsin",
    72: "Puerto Rico"
}
df_2024_data['state'] = df_2024_data['state'].map(state_mapping).fillna(df_2024_data['state'])
print(df_2024_data['state'].head())

0    Illinois
1      Hawaii
2      Alaska
3    Maryland
7       Texas
Name: state, dtype: object


In [34]:
print(df_2024_data.head(10))

      newid  seqno   expname cost_  ref_mo  ref_yr   gift     ucc   cost  \
0   5348484     17   QADOTHX     E       1    2024  False  270310    3.0   
1   5356724     10   QADOTHX     E       2    2024  False  270310    4.0   
2   5357274     14   QADOTHX     E       2    2024  False  270310    3.0   
3   5358004     25   QADOTHX     E       2    2024  False  270310    5.0   
7   5366304     17   QADOTHX     E       1    2024  False  270310   10.0   
9   5366674     25   QADOTHX     E       1    2024  False  270310    4.0   
10  5424513     23   QADOTHX     E       1    2024  False  270310    4.0   
15  5343904     17  TELCEL3X     D       1    2024  False  270102   32.0   
18  5343964     50  TELCEL3X     D       1    2024  False  270102  177.0   
19  5343964     51  TELCEL3X     E       1    2024  False  270102  101.0   

    fam_size  ...  sex_ref  total_expenditure_prior_quarter  \
0          2  ...        1                       10609.3333   
1          1  ...        1           

In [37]:
df_2024_data['division'] = df_2024_data['division'].astype("Int64")
division_dict ={
    1: "New England",
    2: "Middle Atlantic",
    3: "East North Central",
    4: "West North Central",
    5: "South Central",
    6: "East South Central",
    7: "West South Central",
    8: "Mountain",
    9: "Pacific"
}

df_2024_data['division'] = df_2024_data['division'].map(division_dict).fillna(df_2024_data['division'])
print(df_2024_data['division'].head())

ValueError: invalid literal for int() with base 10: 'East North Central'

In [39]:
print(df_2024_data.head(10))

      newid  seqno   expname cost_  ref_mo  ref_yr   gift     ucc   cost  \
0   5348484     17   QADOTHX     E       1    2024  False  270310    3.0   
1   5356724     10   QADOTHX     E       2    2024  False  270310    4.0   
2   5357274     14   QADOTHX     E       2    2024  False  270310    3.0   
3   5358004     25   QADOTHX     E       2    2024  False  270310    5.0   
7   5366304     17   QADOTHX     E       1    2024  False  270310   10.0   
9   5366674     25   QADOTHX     E       1    2024  False  270310    4.0   
10  5424513     23   QADOTHX     E       1    2024  False  270310    4.0   
15  5343904     17  TELCEL3X     D       1    2024  False  270102   32.0   
18  5343964     50  TELCEL3X     D       1    2024  False  270102  177.0   
19  5343964     51  TELCEL3X     E       1    2024  False  270102  101.0   

    fam_size  ...  sex_ref  total_expenditure_prior_quarter  \
0          2  ...        1                       10609.3333   
1          1  ...        1           

In [46]:
df_2024_data['cost'].dropna(inplace=True)
df_2024_data['cost'] = df_2024_data['cost'].round().astype("Int64")


In [48]:
print(df_2024_data.head(10))

      newid  seqno   expname cost_  ref_mo  ref_yr   gift     ucc  cost  \
0   5348484     17   QADOTHX     E       1    2024  False  270310     3   
1   5356724     10   QADOTHX     E       2    2024  False  270310     4   
2   5357274     14   QADOTHX     E       2    2024  False  270310     3   
3   5358004     25   QADOTHX     E       2    2024  False  270310     5   
7   5366304     17   QADOTHX     E       1    2024  False  270310    10   
9   5366674     25   QADOTHX     E       1    2024  False  270310     4   
10  5424513     23   QADOTHX     E       1    2024  False  270310     4   
15  5343904     17  TELCEL3X     D       1    2024  False  270102    32   
18  5343964     50  TELCEL3X     D       1    2024  False  270102   177   
19  5343964     51  TELCEL3X     E       1    2024  False  270102   101   

    fam_size  ...  sex_ref  total_expenditure_prior_quarter  \
0          2  ...        1                       10609.3333   
1          1  ...        1                      

In [49]:
urban_dict ={
    1: "Urban",
    2: "Rural"
}
df_2024_data['urban'] = df_2024_data['urban'].map(urban_dict)

In [52]:
print(df_2024_data.head(5))

     newid  seqno  expname cost_  ref_mo  ref_yr   gift     ucc  cost  \
0  5348484     17  QADOTHX     E       1    2024  False  270310     3   
1  5356724     10  QADOTHX     E       2    2024  False  270310     4   
2  5357274     14  QADOTHX     E       2    2024  False  270310     3   
3  5358004     25  QADOTHX     E       2    2024  False  270310     5   
7  5366304     17  QADOTHX     E       1    2024  False  270310    10   

   fam_size  ...  sex_ref  total_expenditure_prior_quarter  \
0         2  ...        1                       10609.3333   
1         1  ...        1                        4515.7500   
2         1  ...        1                        3666.5000   
3         1  ...        2                        2597.6667   
7         3  ...        2                        3983.3333   

   total_expenditure_current_quarter     state  imputed_income_before_tax  \
0                          6501.1667  Illinois                   170000.0   
1                          8258.50

In [54]:
psu_dict ={
    1102: "Philadelphia – Wilmington – Atlantic City, PA – NJ – DE - MD",
    1103: "Boston – Brockton – Nashua, MA – NH – ME CT",
    1109: "New York, NY",
    1110: "New York, Connecticut suburbs",
    1111: "New Jersey suburbs",
    1207: "Chicago – Gary – Kenosha, IL – IN - WI",
    1208: "Detroit – Ann Arbor – Flint, MI",
    1210: "Cleveland – Akron, OH",
    1211: "Minneapolis – St. Paul, MN – WI",
    1312: "Washington, DC – MD – VA – WV",
    1313: "Baltimore, MD",
    1316: "Dallas – Ft. Worth, TX",
    1318: "Houston – Galveston – Brazoria, TX",
    1319: "Atlanta, GA",
    1320: "Miami – Ft. Lauderdale, FL",
    1419: "Los Angeles – Orange, CA",
    1420: "Los Angeles suburbs, CA",
    1422: "San Francisco – Oakland – San Jose, CA",
    1423: "Seattle – Tacoma – Bremerton, WA",
    1424: "San Diego, CA",
    1429: "Phoenix – Mesa, AZ",
    "S11A": "Boston-Cambridge-Newton, MA-NH",
    "S12A": "New York-Newark-Jersey City, NY-NJ-PA",
    "S12B": "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",
    "S23A": "Chicago-Naperville-Elgin, IL-IN-WI",
    "S23B": "Detroit-Warren-Dearborn, MI",
    "S24A": "Minneapolis-St. Paul-Bloomington, MN-WI",
    "S24B": "St. Louis, MO-IL",
    "S35A": "Washington-Arlington-Alexandria, DC-VA-MD-WV",
    "S35B": "Miami-Fort Lauderdale-West Palm Beach, FL",
    "S35C": "Atlanta-Sandy Springs-Roswell, GA",
    "S35D": "Tampa-St. Petersburg-Clearwater, FL",
    "S35E": "Baltimore-Columbia-Towson, MD",
    "S37A": "Dallas-Fort Worth-Arlington, TX",
    "S37B": "Houston-The Woodlands-Sugar Land, TX",
    "S48A": "Phoenix-Mesa-Scottsdale, AZ",
    "S48B": "Denver-Aurora-Lakewood, CO",
    "S49A": "Los Angeles-Long Beach-Anaheim, CA",
    "S49B": "San Francisco-Oakland-Hayward, CA",
    "S49C": "Riverside-San Bernardino-Ontario, CA",
    "S49D": "Seattle-Tacoma-Bellevue, WA",
    "S49E": "San Diego-Carlsbad, CA",
    "S49F": "Honolulu, HI",
    "S49G": "Anchorage, AK"

}

df_2024_data['psu'] = df_2024_data['psu'].map(psu_dict).fillna(df_2024_data['psu'])

In [59]:
print(df_2024_data.loc[df_2024_data['state'] == "New York"])

         newid  seqno   expname cost_  ref_mo  ref_yr   gift     ucc  cost  \
126    5345484     47  TELCEL3X     D       1    2024  False  270102   130   
127    5345484     48  QADCAB3X     E       1    2024  False  270310    70   
128    5345484     48  QADINE3X     E       1    2024  False  690114    36   
129    5345484     48  TELRES3X     E       1    2024  False  270106    44   
322    5347964     24  TELCEL3X     D       1    2024  False  270102    50   
...        ...    ...       ...   ...     ...     ...    ...     ...   ...   
14762  5432933     74   SUBEXPX     D       1    2024  False  590230    60   
14763  5432933     77   SUBEXPX     D       1    2024  False  310243    40   
14764  5432933     80   SUBEXPX     D       1    2024  False  310243     5   
14765  5432933     81   SUBEXPX     D       1    2024  False  310243     6   
14802  5556924     34   SUBEXPX     D       1    2024  False  310243    15   

       fam_size  ...  sex_ref  total_expenditure_prior_quarter 

In [60]:
df_2024_data['psu_clean'] = (df_2024_data['psu'].astype("string").str.strip().replace({"":pd.NA}))
df_2024_data['sampling_city'] = df_2024_data['psu_clean'].str.split(",",n=1).str[0]
df_2024_data['sampling_state'] = df_2024_data['psu_clean'].str.split(",",n=1).str[1].str.split()
df_2024_data['sampling_primary_states'] = df_2024_data['psu_clean'].str.split("-",n=1).str[0]


In [61]:
print(df_2024_data.head())

     newid  seqno  expname cost_  ref_mo  ref_yr   gift     ucc  cost  \
0  5348484     17  QADOTHX     E       1    2024  False  270310     3   
1  5356724     10  QADOTHX     E       2    2024  False  270310     4   
2  5357274     14  QADOTHX     E       2    2024  False  270310     3   
3  5358004     25  QADOTHX     E       2    2024  False  270310     5   
7  5366304     17  QADOTHX     E       1    2024  False  270310    10   

   fam_size  ...  imputed_income_before_tax  \
0         2  ...                   170000.0   
1         1  ...                    91200.0   
2         1  ...                    83311.0   
3         1  ...                   145058.9   
7         3  ...                    72000.0   

                                    psu            division  urban  \
0    Chicago-Naperville-Elgin, IL-IN-WI  East North Central  Urban   
1                          Honolulu, HI             Pacific  Urban   
2                         Anchorage, AK             Pacific  Urban  

In [62]:
print(df_2024_data['sampling_state'].dtype)

object


In [66]:
df_2024_data.drop(columns=['region'],inplace=True)

In [69]:
print(df_2024_data['fam_size'])

0        2
1        1
2        1
3        1
7        3
        ..
14907    1
14908    1
14909    1
14910    3
14911    3
Name: fam_size, Length: 6820, dtype: int64


In [70]:
df_2024_data.drop(columns=['psu'], inplace=True)

In [71]:
print(df_2024_data.head())

     newid  seqno  expname cost_  ref_mo  ref_yr   gift     ucc  cost  \
0  5348484     17  QADOTHX     E       1    2024  False  270310     3   
1  5356724     10  QADOTHX     E       2    2024  False  270310     4   
2  5357274     14  QADOTHX     E       2    2024  False  270310     3   
3  5358004     25  QADOTHX     E       2    2024  False  270310     5   
7  5366304     17  QADOTHX     E       1    2024  False  270310    10   

   fam_size  ...     state  imputed_income_before_tax            division  \
0         2  ...  Illinois                   170000.0  East North Central   
1         1  ...    Hawaii                    91200.0             Pacific   
2         1  ...    Alaska                    83311.0             Pacific   
3         1  ...  Maryland                   145058.9       South Central   
7         3  ...     Texas                    72000.0  West South Central   

   urban                      product_description  region_description  \
0  Urban  Cable and satel

In [76]:
print(df_2024_data.loc[df_2024_data['newid'] == 5348484,['state','newid','sampling_primary_states']])

         state    newid sampling_primary_states
0     Illinois  5348484                 Chicago
375   Illinois  5348484                 Chicago
376   Illinois  5348484                 Chicago
2542  Illinois  5348484                 Chicago


In [77]:
df_2024_data.drop(columns=['sampling_primary_states','sampling_state','psu_clean'],inplace=True)

In [78]:
print(df_2024_data.head())

     newid  seqno  expname cost_  ref_mo  ref_yr   gift     ucc  cost  \
0  5348484     17  QADOTHX     E       1    2024  False  270310     3   
1  5356724     10  QADOTHX     E       2    2024  False  270310     4   
2  5357274     14  QADOTHX     E       2    2024  False  270310     3   
3  5358004     25  QADOTHX     E       2    2024  False  270310     5   
7  5366304     17  QADOTHX     E       1    2024  False  270310    10   

   fam_size  ...  sex_ref  total_expenditure_prior_quarter  \
0         2  ...        1                       10609.3333   
1         1  ...        1                        4515.7500   
2         1  ...        1                        3666.5000   
3         1  ...        2                        2597.6667   
7         3  ...        2                        3983.3333   

   total_expenditure_current_quarter     state  imputed_income_before_tax  \
0                          6501.1667  Illinois                   170000.0   
1                          8258.50

In [79]:
df_2024_data.rename(columns={"state":"sampling_state"},inplace=True)

In [80]:
print(df_2024_data.head())

     newid  seqno  expname cost_  ref_mo  ref_yr   gift     ucc  cost  \
0  5348484     17  QADOTHX     E       1    2024  False  270310     3   
1  5356724     10  QADOTHX     E       2    2024  False  270310     4   
2  5357274     14  QADOTHX     E       2    2024  False  270310     3   
3  5358004     25  QADOTHX     E       2    2024  False  270310     5   
7  5366304     17  QADOTHX     E       1    2024  False  270310    10   

   fam_size  ...  sex_ref  total_expenditure_prior_quarter  \
0         2  ...        1                       10609.3333   
1         1  ...        1                        4515.7500   
2         1  ...        1                        3666.5000   
3         1  ...        2                        2597.6667   
7         3  ...        2                        3983.3333   

   total_expenditure_current_quarter  sampling_state  \
0                          6501.1667        Illinois   
1                          8258.5000          Hawaii   
2                   

In [83]:
df_2024_data['cost_'].value_counts()

cost_
D    4890
E    1869
F      26
T      22
G      11
U       2
Name: count, dtype: int64

In [84]:
df_2024_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6820 entries, 0 to 14911
Data columns (total 26 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   newid                              6820 non-null   int64  
 1   seqno                              6820 non-null   int64  
 2   expname                            6820 non-null   object 
 3   cost_                              6820 non-null   object 
 4   ref_mo                             6820 non-null   int64  
 5   ref_yr                             6820 non-null   int64  
 6   gift                               6820 non-null   bool   
 7   ucc                                6820 non-null   int64  
 8   cost                               6820 non-null   Int64  
 9   fam_size                           6820 non-null   int64  
 10  family_income_before_tax           6820 non-null   int64  
 11  calibration_weight                 6820 non-null   float64
 

In [85]:
print(df_2024_data.head())

     newid  seqno  expname cost_  ref_mo  ref_yr   gift     ucc  cost  \
0  5348484     17  QADOTHX     E       1    2024  False  270310     3   
1  5356724     10  QADOTHX     E       2    2024  False  270310     4   
2  5357274     14  QADOTHX     E       2    2024  False  270310     3   
3  5358004     25  QADOTHX     E       2    2024  False  270310     5   
7  5366304     17  QADOTHX     E       1    2024  False  270310    10   

   fam_size  ...  sex_ref  total_expenditure_prior_quarter  \
0         2  ...        1                       10609.3333   
1         1  ...        1                        4515.7500   
2         1  ...        1                        3666.5000   
3         1  ...        2                        2597.6667   
7         3  ...        2                        3983.3333   

   total_expenditure_current_quarter  sampling_state  \
0                          6501.1667        Illinois   
1                          8258.5000          Hawaii   
2                   

In [86]:
print(df_2024_data.tail())

         newid  seqno  expname cost_  ref_mo  ref_yr   gift     ucc  cost  \
14907  5605481     36  SUBEXPX     D       1    2024  False  310243    15   
14908  5605481     38  SUBEXPX     D       1    2024  False  620930    11   
14909  5605481     39  SUBEXPX     D       2    2024  False  620930    11   
14910  5605671     26  SUBEXPX     D       2    2024  False  310243    40   
14911  5605671     26  SUBEXPX     D       1    2024  False  310243    40   

       fam_size  ...  sex_ref  total_expenditure_prior_quarter  \
14907         1  ...        1                        6194.5833   
14908         1  ...        1                        6194.5833   
14909         1  ...        1                        6194.5833   
14910         3  ...        1                        9004.9167   
14911         3  ...        1                        9004.9167   

       total_expenditure_current_quarter  sampling_state  \
14907                         11702.1666       Minnesota   
14908               

In [87]:
df_2024_data['product_description'].value_counts()

product_description
Cellular phone service                                     1733
Computer information services (internet)                   1703
Rental, streaming, and downloading videos                  1021
Cable and satellite television services                     862
Residential telephone including VOIP                        428
Streaming and downloading audio                             381
Books, digital books, or book subscriptions                 177
Online gaming services                                       68
Computers and computer hardware for nonbusiness use          41
Internet services away from home                             38
Computer accessories                                         37
Computer software                                            37
Telephones and accessories                                   36
Vehicle parts, accessories, fluid excluding tires            34
Video game software                                          34
Televisions         

In [88]:
df_2024_data.drop(columns=['ucc'],inplace=True)

In [89]:
sex_map ={
    1: "Male",
    2: "Female"
}

df_2024_data['sex_ref'] = df_2024_data['sex_ref'].map(sex_map).fillna(df_2024_data['sex_ref'])

In [90]:
print(df_2024_data.tail())

         newid  seqno  expname cost_  ref_mo  ref_yr   gift  cost  fam_size  \
14907  5605481     36  SUBEXPX     D       1    2024  False    15         1   
14908  5605481     38  SUBEXPX     D       1    2024  False    11         1   
14909  5605481     39  SUBEXPX     D       2    2024  False    11         1   
14910  5605671     26  SUBEXPX     D       2    2024  False    40         3   
14911  5605671     26  SUBEXPX     D       1    2024  False    40         3   

       family_income_before_tax  ...  sex_ref  \
14907                    133700  ...     Male   
14908                    133700  ...     Male   
14909                    133700  ...     Male   
14910                    184200  ...     Male   
14911                    184200  ...     Male   

       total_expenditure_prior_quarter  total_expenditure_current_quarter  \
14907                        6194.5833                         11702.1666   
14908                        6194.5833                         11702.1666   

In [91]:
df_2024_data.rename(columns={"cost_":"cost_flag"},inplace=True)

In [92]:
print(df_2024_data.tail())

         newid  seqno  expname cost_flag  ref_mo  ref_yr   gift  cost  \
14907  5605481     36  SUBEXPX         D       1    2024  False    15   
14908  5605481     38  SUBEXPX         D       1    2024  False    11   
14909  5605481     39  SUBEXPX         D       2    2024  False    11   
14910  5605671     26  SUBEXPX         D       2    2024  False    40   
14911  5605671     26  SUBEXPX         D       1    2024  False    40   

       fam_size  family_income_before_tax  ...  sex_ref  \
14907         1                    133700  ...     Male   
14908         1                    133700  ...     Male   
14909         1                    133700  ...     Male   
14910         3                    184200  ...     Male   
14911         3                    184200  ...     Male   

       total_expenditure_prior_quarter  total_expenditure_current_quarter  \
14907                        6194.5833                         11702.1666   
14908                        6194.5833              

In [93]:
df_2024_data.describe()

,newid,seqno,ref_mo,ref_yr,cost,fam_size,family_income_before_tax,calibration_weight,number_of_earners,popsize,interview_month,interview_year,total_expenditure_prior_quarter,total_expenditure_current_quarter,imputed_income_before_tax
count,6.820000e+03,6820.000000,6820.000000,6820.0,6820.0,6820.000000,6820.000000,6820.000000,6820.000000,6820.000000,6820.000000,6820.0,6820.000000,6820.000000,6820.000000
mean,5.492532e+06,28.272874,1.310850,2024.0,91.71393,2.423460,126679.242229,24417.613587,1.388416,1.657771,2.623607,2024.0,12227.618727,12932.424120,142986.881994
std,9.574756e+04,15.837898,0.462876,0.0,173.112013,1.359494,132673.390761,11769.372724,1.034249,0.719212,0.484516,0.0,13636.270703,11382.021287,131919.949761
min,5.343904e+06,1.000000,1.000000,2024.0,1.0,1.000000,-6083.000000,1592.845000,0.000000,1.000000,2.000000,2024.0,146.500000,73.250000,-26951.900000
25%,5.426493e+06,17.000000,1.000000,2024.0,25.0,1.000000,37556.000000,15765.196000,1.000000,1.000000,2.000000,2024.0,4828.416700,5820.333300,49364.900000
50%,5.551652e+06,26.000000,1.000000,2024.0,62.0,2.000000,87176.000000,24177.552000,1.000000,2.000000,3.000000,2024.0,8388.500000,9550.133300,104488.200000
75%,5.586774e+06,36.000000,2.000000,2024.0,107.0,3.000000,169000.000000,31268.448000,2.000000,2.000000,3.000000,2024.0,14617.500100,15536.166700,188000.000000
max,5.607981e+06,120.000000,2.000000,2024.0,6000.0,10.000000,800733.000000,85737.536000,7.000000,4.000000,3.000000,2024.0,204944.087500,115150.400000,949344.200000


In [95]:
flag_desc = {
    "A": "Valid blank (not anticipated)",
    "B": "Invalid blank (invalid nonresponse)",
    "C": "Blank (don't know/refusal/other)",
    "D": "Valid value (unadjusted)",
    "E": "Valid value (allocated)",
    "F": "Valid value (imputed/adjusted)",
    "G": "Valid value (allocated + imputed/adjusted)",
    "H": "Valid blank (parent record allocated elsewhere)",
    "T": "Valid value (topcoded/suppressed)",
    "U": "Valid value (allocated then topcoded/suppressed)",
    "V": "Valid value (imputed/adjusted then topcoded/suppressed)",
    "W": "Valid value (allocated + imputed/adjusted then topcoded/suppressed)",
}
df_2024_data['cost_flag'] = df_2024_data['cost_flag'].astype("string").str.strip().str.upper()
df_2024_data['cost_flag'] = df_2024_data['cost_flag'].map(flag_desc)

In [99]:
df_2024_data.drop(columns=['Unnamed: 0'], errors="ignore",inplace=True)
print(df_2024_data.head())

     newid  seqno  expname                cost_flag  ref_mo  ref_yr   gift  \
0  5348484     17  QADOTHX  Valid value (allocated)       1    2024  False   
1  5356724     10  QADOTHX  Valid value (allocated)       2    2024  False   
2  5357274     14  QADOTHX  Valid value (allocated)       2    2024  False   
3  5358004     25  QADOTHX  Valid value (allocated)       2    2024  False   
7  5366304     17  QADOTHX  Valid value (allocated)       1    2024  False   

   cost  fam_size  family_income_before_tax  ...  sex_ref  \
0     3         2                    170000  ...     Male   
1     4         1                     91200  ...     Male   
2     3         1                     83311  ...     Male   
3     5         1                    181256  ...   Female   
7    10         3                     72000  ...   Female   

   total_expenditure_prior_quarter  total_expenditure_current_quarter  \
0                       10609.3333                          6501.1667   
1                 

In [101]:
df_2024_data.columns

Index(['newid', 'seqno', 'expname', 'cost_flag', 'ref_mo', 'ref_yr', 'gift',
       'cost', 'fam_size', 'family_income_before_tax', 'calibration_weight',
       'number_of_earners', 'popsize', 'interview_month', 'interview_year',
       'sex_ref', 'total_expenditure_prior_quarter',
       'total_expenditure_current_quarter', 'sampling_state',
       'imputed_income_before_tax', 'division', 'urban', 'product_description',
       'region_description', 'sampling_city'],
      dtype='object')

In [102]:
population_dict ={
    1: "5+ Million",
    2: "1-5 Million",
    3: "0.5-1.0 Million",
    4: "100-500 Thousands",
    5: "100- Thousands",
    6: "Suppressed"
}
df_2024_data['Population_bucket'] = pd.Categorical(df_2024_data['popsize'].map(population_dict))

In [103]:
print(df_2024_data.tail())

         newid  seqno  expname                 cost_flag  ref_mo  ref_yr  \
14907  5605481     36  SUBEXPX  Valid value (unadjusted)       1    2024   
14908  5605481     38  SUBEXPX  Valid value (unadjusted)       1    2024   
14909  5605481     39  SUBEXPX  Valid value (unadjusted)       2    2024   
14910  5605671     26  SUBEXPX  Valid value (unadjusted)       2    2024   
14911  5605671     26  SUBEXPX  Valid value (unadjusted)       1    2024   

        gift  cost  fam_size  family_income_before_tax  ...  \
14907  False    15         1                    133700  ...   
14908  False    11         1                    133700  ...   
14909  False    11         1                    133700  ...   
14910  False    40         3                    184200  ...   
14911  False    40         3                    184200  ...   

       total_expenditure_prior_quarter  total_expenditure_current_quarter  \
14907                        6194.5833                         11702.1666   
14908     

In [104]:
df_2024_data.drop(columns=['popsize'],inplace=True)

In [105]:
df_2024_data.head()

,newid,seqno,expname,cost_flag,ref_mo,ref_yr,gift,cost,fam_size,family_income_before_tax,...,total_expenditure_prior_quarter,total_expenditure_current_quarter,sampling_state,imputed_income_before_tax,division,urban,product_description,region_description,sampling_city,Population_bucket
0,5348484,17,QADOTHX,Valid value (allocated),1,2024,False,3,2,170000,...,10609.3333,6501.1667,Illinois,170000.0,East North Central,Urban,Cable and satellite television services,Midwest,Chicago-Naperville-Elgin,5+ Million
1,5356724,10,QADOTHX,Valid value (allocated),2,2024,False,4,1,91200,...,4515.7500,8258.5000,Hawaii,91200.0,Pacific,Urban,Cable and satellite television services,West,Honolulu,0.5-1.0 Million
2,5357274,14,QADOTHX,Valid value (allocated),2,2024,False,3,1,83311,...,3666.5000,9154.0000,Alaska,83311.0,Pacific,Urban,Cable and satellite television services,West,Anchorage,100-500 Thousands
3,5358004,25,QADOTHX,Valid value (allocated),2,2024,False,5,1,181256,...,2597.6667,12292.3333,Maryland,145058.9,South Central,Urban,Cable and satellite television services,South,Baltimore-Columbia-Towson,1-5 Million
7,5366304,17,QADOTHX,Valid value (allocated),1,2024,False,10,3,72000,...,3983.3333,8911.6667,Texas,72000.0,West South Central,Urban,Cable and satellite television services,South,Houston-The Woodlands-Sugar Land,5+ Million


In [106]:
df_2024_data.to_csv('./cleaned_2024_digital_products_expenditure_dataset.csv')


In [107]:
df_2024_data.columns

Index(['newid', 'seqno', 'expname', 'cost_flag', 'ref_mo', 'ref_yr', 'gift',
       'cost', 'fam_size', 'family_income_before_tax', 'calibration_weight',
       'number_of_earners', 'interview_month', 'interview_year', 'sex_ref',
       'total_expenditure_prior_quarter', 'total_expenditure_current_quarter',
       'sampling_state', 'imputed_income_before_tax', 'division', 'urban',
       'product_description', 'region_description', 'sampling_city',
       'Population_bucket'],
      dtype='object')

In [110]:
print(df_2024_data.head(1))

     newid  seqno  expname                cost_flag  ref_mo  ref_yr   gift  \
0  5348484     17  QADOTHX  Valid value (allocated)       1    2024  False   

   cost  fam_size  family_income_before_tax  ...  \
0     3         2                    170000  ...   

   total_expenditure_prior_quarter  total_expenditure_current_quarter  \
0                       10609.3333                          6501.1667   

   sampling_state  imputed_income_before_tax            division  urban  \
0        Illinois                   170000.0  East North Central  Urban   

                       product_description region_description  \
0  Cable and satellite television services            Midwest   

              sampling_city Population_bucket  
0  Chicago-Naperville-Elgin        5+ Million  

[1 rows x 25 columns]


In [111]:
df_2024 = pd.read_csv('cleaned_2024_digital_products_expenditure_dataset.csv')

In [112]:
df_2024.head()

,Unnamed: 0,newid,seqno,expname,cost_flag,ref_mo,ref_yr,gift,cost,fam_size,...,total_expenditure_prior_quarter,total_expenditure_current_quarter,sampling_state,imputed_income_before_tax,division,urban,product_description,region_description,sampling_city,Population_bucket
0,0,5348484,17,QADOTHX,Valid value (allocated),1,2024,False,3,2,...,10609.3333,6501.1667,Illinois,170000.0,East North Central,Urban,Cable and satellite television services,Midwest,Chicago-Naperville-Elgin,5+ Million
1,1,5356724,10,QADOTHX,Valid value (allocated),2,2024,False,4,1,...,4515.7500,8258.5000,Hawaii,91200.0,Pacific,Urban,Cable and satellite television services,West,Honolulu,0.5-1.0 Million
2,2,5357274,14,QADOTHX,Valid value (allocated),2,2024,False,3,1,...,3666.5000,9154.0000,Alaska,83311.0,Pacific,Urban,Cable and satellite television services,West,Anchorage,100-500 Thousands
3,3,5358004,25,QADOTHX,Valid value (allocated),2,2024,False,5,1,...,2597.6667,12292.3333,Maryland,145058.9,South Central,Urban,Cable and satellite television services,South,Baltimore-Columbia-Towson,1-5 Million
4,7,5366304,17,QADOTHX,Valid value (allocated),1,2024,False,10,3,...,3983.3333,8911.6667,Texas,72000.0,West South Central,Urban,Cable and satellite television services,South,Houston-The Woodlands-Sugar Land,5+ Million


In [114]:
month_map ={
    1 : "January",
    2 : "February",
    3 : "March",
    4 : "April",
    5 : "May",
    6 : "June",
    7 : "July",
    8 : "August",
    9 : "September",
    10 : "October",
    11 : "November",
    12 : "December"
}
df_2024['interview_month'] = df_2024['interview_month'].map(month_map).fillna(df_2024['interview_month'])

In [115]:
df_2024['interview_month']

0       February
1          March
2          March
3          March
4          March
          ...   
6815       March
6816       March
6817       March
6818       March
6819       March
Name: interview_month, Length: 6820, dtype: object

In [116]:
df_2024['ref_mo'] = df_2024['ref_mo'].map(month_map).fillna(df_2024['ref_mo'])

In [117]:
df_2024['ref_mo']

0        January
1       February
2       February
3       February
4        January
          ...   
6815     January
6816     January
6817    February
6818    February
6819     January
Name: ref_mo, Length: 6820, dtype: object

In [118]:
df_2024.columns

Index(['Unnamed: 0', 'newid', 'seqno', 'expname', 'cost_flag', 'ref_mo',
       'ref_yr', 'gift', 'cost', 'fam_size', 'family_income_before_tax',
       'calibration_weight', 'number_of_earners', 'interview_month',
       'interview_year', 'sex_ref', 'total_expenditure_prior_quarter',
       'total_expenditure_current_quarter', 'sampling_state',
       'imputed_income_before_tax', 'division', 'urban', 'product_description',
       'region_description', 'sampling_city', 'Population_bucket'],
      dtype='object')

In [119]:
df_2024.to_csv("cleaned_2024_digital_products_expenditure_dataset.csv")